# Таблицы MCC и типов карт (DRP) — по периметру дашборда

Листы дашборда «MCC-коды» и «Терминалы» читают две отдельные таблицы, которые DAG не пополняет:

| Таблица | Зерно |
|---|---|
| `sbx_da.tmp_shestopalov_acq_mcc_month` | месяц × MCC |
| `sbx_da.tmp_shestopalov_acq_card_type_month` | месяц × MCC × платёжная система |

**Периметр — как в DAG** (`acq_01_sa_perimeter.sql`, `acq_05_trx_agr.sql`): договор торгового эквайринга (`acq_class = 'SA'`), действует в этом месяце, есть терминальное условие типа `P` в этом месяце, договор и клиент не удалены, у клиента есть ИНН. Операции — те же фильтры, что в DAG. Поэтому оборот на листе MCC совпадает с оборотом на остальных листах.

Один запрос в Impala на месяц даёт обе таблицы (месяц × MCC × тип карты). Перед записью — **сверка оборота с основной таблицей дашборда**: если месяц расходится больше чем на `MAX_DIFF_PCT`, запись не выполняется. Перед удалением старых строк таблицы **сохраняются в резервную копию** (`Data/mcc_card_type_append/backup_*.csv.gz`).

В DRP **удаляются только месяцы `MONTHS` и дописываются заново**; сами таблицы не пересоздаются (без DROP).

Как запускать: Kernel → Restart Kernel and Run All Cells. Спросит логин keytab (Impala) и логин/пароль DRP. Если LDAP DRP отказал, не повторяйте больше 2 раз.

Каждый месяц — тяжёлый запрос к `scd1_trx` (15–25 минут). Посчитанные закрытые месяцы сохраняются в `Data/mcc_card_type_append/perim_*.csv.gz` — если прогон прервётся, повторный запуск их не пересчитывает. Незакрытый месяц (`OPEN_MONTH`) пересчитывается всегда.

Каждый следующий месяц: `MONTHS = ['2026-11', '2026-12']`, `OPEN_MONTH = '2026-12'`.

In [ ]:
import getpass
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 100)

MONTHS = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06',
          '2026-07', '2026-08', '2026-09', '2026-10', '2026-11']
OPEN_MONTH = '2026-11'  # незакрытый месяц: всегда считать заново
MAX_DIFF_PCT = 2.0      # допустимое расхождение оборота с основной таблицей, %
FORCE_WRITE = False     # True — записать, даже если сверка не сошлась (только осознанно)

DRP_SCHEMA = 'sbx_da'
MCC_TABLE = 'tmp_shestopalov_acq_mcc_month'
CARD_TABLE = 'tmp_shestopalov_acq_card_type_month'
FIN_TABLE = 'sbx_da.tmp_shestopalov_acq_fin_version_test'

MEM_LIMIT_FALLBACKS = ['8g', '4g']
FETCH_RETRIES = 3
IMPALA_USER_DEFAULT = 'Shestopalov-VYur'

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd()
CKPT_DIR = DATA_DIR / 'mcc_card_type_append'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert MONTHS == sorted(MONTHS) and all(re.fullmatch(r'\d{4}-\d{2}', m) for m in MONTHS)


def month_bounds(label):
    start = pd.Timestamp(label + '-01')
    end = start + pd.offsets.MonthBegin(1)
    last = end - pd.Timedelta(days=1)
    return start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d'), last.strftime('%Y-%m-%d')


print('MONTHS =', MONTHS, '| OPEN_MONTH =', OPEN_MONTH)
print('checkpoints =', CKPT_DIR)

if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': input('Login (keytab): ').strip() or IMPALA_USER_DEFAULT},
    )
    imp._init_connection()
print('Impala: OK')

## 1) Запрос по периметру DAG и загрузка с повтором

Результат запроса: месяц × тип карты (эмитент) × MCC — операции, оборот, комиссия. Оборот суммируется по парам «операция × договор», как в `acq_05_trx_agr.sql`.

In [ ]:
def sql_month(month_start, month_end_exclusive, month_last):
    return f'''
WITH fiid_rshb AS (
  SELECT DISTINCT CAST(fa.c_fiid AS STRING) AS c_fiid
  FROM ods_alpha.scd1_base24_fiids fa
  WHERE COALESCE(CAST(fa.c_fiid_grp AS STRING), 'UNKNOWN') = 'RSHB'
),
fiid_desc AS (
  SELECT CAST(fi.c_fiid AS STRING) AS c_fiid, MAX(CAST(fi.c_fiid_desc AS STRING)) AS c_fiid_desc
  FROM ods_alpha.scd1_base24_fiids fi
  GROUP BY CAST(fi.c_fiid AS STRING)
),
perim AS (
  SELECT DISTINCT CAST(a.n_agr AS STRING) AS n_agr
  FROM ods_alpha.scd1_agreements a
  JOIN ods_alpha.scd1_companies c ON c.n_cmp = a.n_cmp_client
  LEFT SEMI JOIN (
    SELECT DISTINCT CAST(t.n_agr AS STRING) AS n_agr
    FROM ods_alpha.scd1_agr_terms t
    WHERE UPPER(TRIM(CAST(t.cf_ter_type AS STRING))) = 'P'
      AND COALESCE(t.ods_deleted_flg, '0') <> '1'
      AND CAST(t.d_valid_from AS DATE) <= CAST('{month_last}' AS DATE)
      AND (t.d_valid_to IS NULL OR CAST(t.d_valid_to AS DATE) >= CAST('{month_start}' AS DATE))
  ) pt ON pt.n_agr = CAST(a.n_agr AS STRING)
  WHERE UPPER(TRIM(CAST(a.acq_class AS STRING))) = 'SA'
    AND COALESCE(a.ods_deleted_flg, '0') <> '1'
    AND COALESCE(c.ods_deleted_flg, '0') <> '1'
    AND c.c_inn IS NOT NULL
    AND CAST(a.d_valid_from AS DATE) <= CAST('{month_last}' AS DATE)
    AND (a.d_valid_to IS NULL OR CAST(a.d_valid_to AS DATE) >= CAST('{month_start}' AS DATE))
),
trx_base_raw AS (
  SELECT CAST(t.n_trx AS STRING) AS n_trx, CAST(t.n_mcc AS STRING) AS mcc,
         CAST(t.c_fiid_iss AS STRING) AS c_fiid_iss, CAST(t.n_amt_src AS DOUBLE) AS n_amt_src
  FROM ods_alpha.scd1_trx t
  JOIN fiid_rshb fr ON fr.c_fiid = CAST(t.c_fiid_acq AS STRING)
  WHERE CAST(t.d_trx_orig AS TIMESTAMP) >= CAST('{month_start}' AS TIMESTAMP)
    AND CAST(t.d_trx_orig AS TIMESTAMP) < CAST('{month_end_exclusive}' AS TIMESTAMP)
    AND t.c_nter IS NOT NULL
    AND COALESCE(t.ods_deleted_flg, '0') <> '1'
    AND t.c_trx_class = 'SA'
    AND t.c_trx_type = 'S01'
    AND COALESCE(t.cf_trx_stat, '') <> 'R'
),
trx_base AS (
  SELECT n_trx, MAX(mcc) AS mcc, MAX(c_fiid_iss) AS c_fiid_iss, MAX(n_amt_src) AS n_amt_src
  FROM trx_base_raw GROUP BY n_trx
),
ta AS (
  SELECT CAST(a.n_trx AS STRING) AS n_trx, CAST(a.n_agr AS STRING) AS n_agr,
         MAX(COALESCE(CAST(a.n_amt_tax AS DOUBLE), 0.0)) AS n_amt_tax
  FROM ods_alpha.scd1_trx_acq a
  JOIN trx_base tb ON tb.n_trx = CAST(a.n_trx AS STRING)
  JOIN perim p ON p.n_agr = CAST(a.n_agr AS STRING)
  GROUP BY CAST(a.n_trx AS STRING), CAST(a.n_agr AS STRING)
)
SELECT
  COALESCE(NULLIF(TRIM(fd.c_fiid_desc), ''), 'empty') AS card_type,
  COALESCE(NULLIF(TRIM(tb.mcc), ''), 'empty') AS mcc,
  COUNT(DISTINCT tb.n_trx) AS trx_cnt,
  SUM(tb.n_amt_src) AS trx_sum,
  SUM(ta.n_amt_tax) AS commission_from_ops
FROM ta
JOIN trx_base tb ON tb.n_trx = ta.n_trx
LEFT JOIN fiid_desc fd ON fd.c_fiid = tb.c_fiid_iss
GROUP BY 1, 2
'''


def _ckpt(kind, label):
    return CKPT_DIR / f'{kind}_{label}.csv.gz'


def _is_admission_fail(exc):
    s = str(exc).lower()
    return any(x in s for x in ('not enough memory', 'admission', 'queued reason', 'exceeded timeout', 's00051',
                                'memory limit exceeded'))


def fetch_months(kind, sql_builder):
    parts = []
    for label in MONTHS:
        path = _ckpt(kind, label)
        if path.exists() and label != OPEN_MONTH:
            part = pd.read_csv(path, compression='gzip', dtype={'mcc': str, 'card_type': str})
            print(f'  {kind} {label}: из сохранённого, строк {len(part):,}')
        else:
            m_start, m_end, m_last = month_bounds(label)
            part, last_exc = None, None
            for attempt in range(1, FETCH_RETRIES + 1):
                mem = MEM_LIMIT_FALLBACKS[min(attempt - 1, len(MEM_LIMIT_FALLBACKS) - 1)]
                print(f'  {kind} {label}: Impala, mem={mem}, попытка {attempt}/{FETCH_RETRIES}... ({datetime.now():%H:%M})', flush=True)
                try:
                    with imp:
                        try:
                            imp.execute(f'set MEM_LIMIT={mem}')
                        except Exception:
                            pass
                        part = imp.fetch(sql_builder(m_start, m_end, m_last))
                    break
                except Exception as exc:
                    last_exc = exc
                    print(f'  {kind} {label}: ошибка {type(exc).__name__}: {str(exc)[:300]}')
                    if not (_is_admission_fail(exc) and attempt < FETCH_RETRIES):
                        break
            if part is None:
                raise RuntimeError(f'{kind} {label}: не удалось получить из Impala') from last_exc
            if len(part) == 0:
                raise RuntimeError(f'{kind} {label}: Impala вернула 0 строк — месяц не грузим')
            part = part.copy()
            part.to_csv(path, index=False, compression='gzip')
            print(f'  {kind} {label}: строк {len(part):,}, сохранено {path.name} ({datetime.now():%H:%M})')
        part['report_month'] = label
        parts.append(part)
    raw = pd.concat(parts, ignore_index=True)
    raw['mcc'] = raw['mcc'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    raw['card_type'] = raw['card_type'].astype(str).str.strip()
    for c in ['trx_cnt', 'trx_sum', 'commission_from_ops']:
        raw[c] = pd.to_numeric(raw[c], errors='coerce').fillna(0.0)
    return raw


perim_raw = fetch_months('perim', sql_month)
print('Всего строк:', f'{len(perim_raw):,}')

## 2) MCC по месяцам

Колонки как раньше. Доли считаются внутри месяца. Операции без MCC в таблицу MCC не попадают (как раньше), их оборот показан отдельно.

In [ ]:
NO_MCC = {'empty', '', 'nan', 'None'}
no_mcc = perim_raw[perim_raw['mcc'].isin(NO_MCC)].groupby('report_month')['trx_sum'].sum()
mcc_raw = perim_raw[~perim_raw['mcc'].isin(NO_MCC)]

g = (
    mcc_raw.groupby(['report_month', 'mcc'], as_index=False)
    .agg(trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum'), commission_from_ops=('commission_from_ops', 'sum'))
)
g['snapshot_month_start'] = g['report_month'] + '-01'
month_trx = g.groupby('report_month')['trx_sum'].transform('sum')
month_cnt = g.groupby('report_month')['trx_cnt'].transform('sum')
g['acq_pct'] = np.where(g['trx_sum'] == 0, np.nan, g['commission_from_ops'] / g['trx_sum'] * 100.0)
g['share_trx_sum_pct'] = np.where(month_trx == 0, np.nan, g['trx_sum'] / month_trx * 100.0)
g['share_trx_cnt_pct'] = np.where(month_cnt == 0, np.nan, g['trx_cnt'] / month_cnt * 100.0)
mcc_month_df = g[[
    'report_month', 'snapshot_month_start', 'mcc', 'trx_cnt', 'trx_sum', 'commission_from_ops',
    'acq_pct', 'share_trx_sum_pct', 'share_trx_cnt_pct',
]].sort_values(['report_month', 'trx_sum'], ascending=[True, False]).reset_index(drop=True)

summary = mcc_month_df.groupby('report_month').agg(
    mcc_n=('mcc', 'nunique'), trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum'), share_sum=('share_trx_sum_pct', 'sum')
)
summary['trx_sum_no_mcc'] = no_mcc.reindex(summary.index).fillna(0)
print('Норма: ~135–150 MCC в месяц, сумма долей ~100, оборот без MCC — близко к 0')
display(summary)

## 3) Типы карт по месяцам

Классификация платёжной системы — как в `card_type_month_upload_jan_aug.ipynb`.

In [ ]:
EMPTY_LABELS = {'none', 'nan', 'null', 'unknown', 'empty'}


def _norm_desc(text):
    return '' if pd.isna(text) else str(text).strip().lower()


def classify_scheme_raw(text):
    s = _norm_desc(text)
    if not s or s in EMPTY_LABELS:
        return None
    if re.search(r'union\s*pay|unionpay|юнион', s):
        return 'UnionPay'
    if re.search(r'master\s*card|mastercard|\bmaestro\b', s):
        return 'Mastercard'
    if re.search(r'\bvisa\b|виза', s):
        return 'Visa'
    if re.search(r'(^|[^a-zа-я])мир([^a-zа-я]|$)|nspk\s*mir|\bmir\b', s):
        return 'МИР'
    if re.search(r'\bjcb\b', s):
        return 'JCB'
    if re.search(r'amex|american\s*express', s):
        return 'Amex'
    if re.search(r'\bsbp\b|сбп', s):
        return 'СБП'
    return None


def is_rshb_issuer(text):
    s = _norm_desc(text)
    if not s or s in EMPTY_LABELS:
        return False
    if re.search(r'рсхб|rshb|россельхоз', s):
        return True
    if re.search(r'\bрф\b|российский\s*филиал', s) or s.endswith('рф'):
        return True
    if re.search(r'головн\w*\s+офис|\bцрмб\b', s):
        return True
    return False


def classify_payment_system(text):
    s = _norm_desc(text)
    if not s or s in EMPTY_LABELS:
        return 'прочие'
    raw = classify_scheme_raw(text)
    if raw in ('UnionPay', 'Mastercard', 'Visa', 'МИР'):
        return raw
    if raw in ('JCB', 'Amex', 'СБП'):
        return 'прочие'
    if is_rshb_issuer(text):
        return 'Карта РСХБ'
    return 'прочие'


card_raw = perim_raw.copy()
card_raw['payment_system'] = card_raw['card_type'].map(classify_payment_system)
card_layer = (
    card_raw.groupby(['report_month', 'mcc', 'payment_system'], dropna=False)
    .agg(trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum'), commission_from_ops=('commission_from_ops', 'sum'))
    .reset_index()
)[['report_month', 'mcc', 'payment_system', 'trx_cnt', 'trx_sum', 'commission_from_ops']]

print('Норма: ~530–580 строк в месяц')
display(card_layer.groupby('report_month').agg(rows=('mcc', 'size'), trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum')))
display(card_layer.pivot_table(index='payment_system', columns='report_month', values='trx_sum', aggfunc='sum').round(0))

## 4) Сверка с основной таблицей дашборда (до записи)

Оборот и комиссия по месяцам: новая таблица MCC (+ операции без MCC) против `tmp_shestopalov_acq_fin_version_test`. Небольшая разница возможна (операция, привязанная к двум договорам, в DAG считается дважды). Если какой-то месяц расходится больше чем на `MAX_DIFF_PCT`, следующая ячейка **не будет записывать** — пришлите скриншот этой таблицы.

In [ ]:
if 'drp' not in globals() or drp is None:
    drp = connect(
        to='DRP',
        user_params={'user_name': input('DRP user: ').strip(), 'password': getpass.getpass('DRP password: ')},
    )

with drp:
    fin = drp.fetch('SELECT CAST(report_month AS TEXT) AS report_month, CAST(trx_sum AS TEXT) AS trx_sum, '
                    'CAST(commission_from_ops AS TEXT) AS commission_from_ops FROM ' + FIN_TABLE)
fin.columns = [str(c).strip().lower() for c in fin.columns]
fin['report_month'] = fin['report_month'].astype(str).str.strip().str[:7]
for c in ['trx_sum', 'commission_from_ops']:
    fin[c] = pd.to_numeric(fin[c], errors='coerce').fillna(0)
fin_m = fin.groupby('report_month')[['trx_sum', 'commission_from_ops']].sum()

new_m = perim_raw.groupby('report_month')[['trx_sum', 'commission_from_ops']].sum()
recon = pd.DataFrame({
    'Оборот: новая MCC, млн': new_m['trx_sum'] / 1e6,
    'Оборот: основная таблица, млн': fin_m['trx_sum'].reindex(new_m.index) / 1e6,
    'Комиссия: новая MCC, млн': new_m['commission_from_ops'] / 1e6,
    'Комиссия: основная таблица, млн': fin_m['commission_from_ops'].reindex(new_m.index) / 1e6,
})
recon['Разница оборота, %'] = (recon.iloc[:, 0] / recon.iloc[:, 1] - 1) * 100
recon['Разница комиссии, %'] = (recon.iloc[:, 2] / recon.iloc[:, 3] - 1) * 100
display(recon.round(2))

bad = recon[(recon['Разница оборота, %'].abs() > MAX_DIFF_PCT) | recon['Разница оборота, %'].isna()]
bad = bad[bad.index != OPEN_MONTH]
RECON_OK = bad.empty
if RECON_OK:
    print(f'Сверка сошлась: все закрытые месяцы в пределах ±{MAX_DIFF_PCT}%')
else:
    print(f'ВНИМАНИЕ: месяцы с расхождением больше ±{MAX_DIFF_PCT}%: {bad.index.tolist()} — запись не будет выполнена (FORCE_WRITE=False)')
print(f'{OPEN_MONTH} не проверяется строго: основная таблица и Impala могли быть посчитаны в разное время')

## 5) Запись в DRP: резервная копия, удалить эти месяцы и дописать

Колонки берутся в порядке таблицы в DRP. Названия MCC (`mcc_name_ru`, `mcc_label`) берутся из того, что уже было в таблице. Если в расчёте нет какой-то колонки таблицы — ячейка остановится до удаления.

In [ ]:
MONTH_SQL = 'SUBSTRING(BTRIM(CAST(report_month AS TEXT)) FROM 1 FOR 7)'

if not RECON_OK and not FORCE_WRITE:
    raise RuntimeError('Сверка с основной таблицей не сошлась — в DRP ничего не записано. Пришлите скриншот раздела 4.')


def to_text_df(df):
    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]
    for c in out.columns:
        out[c] = out[c].map(lambda x: None if pd.isna(x) else str(x)).astype(object)
    return out


def replace_months(table, df):
    fq = f'{DRP_SCHEMA}.{table}'
    up = to_text_df(df)
    months = sorted(up['report_month'].str[:7].unique().tolist())
    assert months == MONTHS, f'{table}: в расчёте месяцы {months}, ожидались {MONTHS}'
    by_month = f'SELECT {MONTH_SQL} AS m, COUNT(*) AS n FROM {fq} GROUP BY 1 ORDER BY 1'
    months_in = ', '.join(f"'{m}'" for m in months)
    with drp:
        db_cols = drp.fetch(
            'SELECT column_name FROM information_schema.columns '
            f"WHERE table_schema = '{DRP_SCHEMA}' AND table_name = '{table}' ORDER BY ordinal_position"
        )['column_name'].tolist()
        label_cols = [c for c in ('mcc_name_ru', 'mcc_label') if c in db_cols and c not in up.columns]
        if label_cols:
            names = drp.fetch(
                f"SELECT REGEXP_REPLACE(BTRIM(CAST(mcc AS TEXT)), '\\.0$', '') AS mcc_key, "
                + ', '.join(f'MAX(NULLIF(BTRIM(CAST({c} AS TEXT)), \'\')) AS {c}' for c in label_cols)
                + f' FROM {fq} GROUP BY 1'
            )
            key = up['mcc'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            for c in label_cols:
                up[c] = key.map(dict(zip(names['mcc_key'], names[c])))
            if 'mcc_name_ru' in label_cols:
                up['mcc_name_ru'] = up['mcc_name_ru'].fillna('MCC ' + key)
            if 'mcc_label' in label_cols:
                nm = up['mcc_name_ru'] if 'mcc_name_ru' in up.columns else ('MCC ' + key)
                up['mcc_label'] = up['mcc_label'].fillna(key + ' — ' + nm.astype(str))
            new_codes = sorted(set(key) - set(names['mcc_key']))
            print(f'{fq}: названия MCC взяты из прежних данных; новых кодов без названия: {len(new_codes)} {new_codes[:20]}')
        missing = [c for c in db_cols if c not in up.columns]
        if not db_cols or missing:
            raise RuntimeError(f'{fq}: нет таблицы или в расчёте нет колонок {missing}. Ничего не удалено.')
        backup = drp.fetch(f'SELECT * FROM {fq}')
        bpath = CKPT_DIR / f'backup_{table}_{datetime.now():%Y%m%d_%H%M}.csv.gz'
        backup.to_csv(bpath, index=False, compression='gzip')
        print(f'{fq}: резервная копия {len(backup):,} строк -> {bpath}')
        before = drp.fetch(by_month)
        drp.execute(f'DELETE FROM {fq} WHERE {MONTH_SQL} IN ({months_in})')
        drp.write(table=fq, df=up[db_cols], mode='append')
        after = drp.fetch(by_month)
    res = before.merge(after, on='m', how='outer', suffixes=('_before', '_after')).sort_values('m')
    print(f'=== {fq}: записано строк {len(up):,} за {months} ===')
    display(res)
    old = res[~res['m'].isin(months)]
    if not (old['n_before'].astype(float) == old['n_after'].astype(float)).all():
        print('ВНИМАНИЕ: изменились строки других месяцев, пришлите скриншот')
    return res


res_mcc = replace_months(MCC_TABLE, mcc_month_df)
res_card = replace_months(CARD_TABLE, card_layer)
print('ГОТОВО. Дашборд: Ctrl+F5 и ⋮ → Обновить дашборд.')